# AEGIS Configuration and Project Setup

Project: Autonomous Energy Grid Intelligence System (AEGIS) — Edge AI based renewable energy monitoring, anomaly detection, federated learning evaluation, and ESP32 deployment.

# 00_config

This notebook initializes the AEGIS project environment and defines the common configuration used throughout the experimental workflow.

This notebook contains:
- project directory setup
- dataset and model path discovery
- reproducibility settings
- utility functions for saving outputs
- experiment metadata used by subsequent notebooks

In [15]:
from pathlib import Path
import json
import os
import random
from datetime import datetime

import numpy as np
import pandas as pd

In [16]:
PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
PROCESSED_DIR = DATA_DIR / "processed"

for p in [DATA_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR, PROCESSED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR     =", DATA_DIR)
print("MODELS_DIR   =", MODELS_DIR)
print("RESULTS_DIR  =", RESULTS_DIR)
print("FIGURES_DIR  =", FIGURES_DIR)
print("TABLES_DIR   =", TABLES_DIR)

PROJECT_ROOT = C:\Users\MOINODHEEN\Moinu\Aegis_Project
DATA_DIR     = C:\Users\MOINODHEEN\Moinu\Aegis_Project\data
MODELS_DIR   = C:\Users\MOINODHEEN\Moinu\Aegis_Project\models
RESULTS_DIR  = C:\Users\MOINODHEEN\Moinu\Aegis_Project\results
FIGURES_DIR  = C:\Users\MOINODHEEN\Moinu\Aegis_Project\figures
TABLES_DIR   = C:\Users\MOINODHEEN\Moinu\Aegis_Project\tables


In [17]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("Random seed set to", SEED)

Random seed set to 42


In [18]:
def find_first(patterns, search_root):
    for pattern in patterns:
        matches = list(search_root.rglob(pattern))
        if matches:
            return matches[0]
    return None

In [19]:
PATHS = {
    "wind_feature_description": find_first(["feature_description*.csv"], DATA_DIR),
    "wind_event_info": find_first(["event_info*.csv"], DATA_DIR),
    "wind_dataset_dir": find_first(["datasets"], DATA_DIR),

    "vestas_excel": find_first(["*.xlsx"], DATA_DIR / "Vestas V52") if (DATA_DIR / "Vestas V52").exists() else find_first(["*Vestas*.xlsx"], DATA_DIR),

    "pv_yield_excel": find_first(["*.xlsx"], DATA_DIR / "PV Yield Forecasting") if (DATA_DIR / "PV Yield Forecasting").exists() else find_first(["*pv*.xlsx", "*solar*.xlsx"], DATA_DIR),

    "pv_fault_amb_mat": find_first(["*amb*.mat", "*Ambient*.mat"], DATA_DIR),
    "pv_fault_elec_mat": find_first(["*elec*.mat", "*Electrical*.mat"], DATA_DIR),

    "node1_log": find_first(["node1_log*.csv"], DATA_DIR),
    "node1_log_clean": find_first(["node1_log_clean*.csv"], DATA_DIR),
    "node2_log": find_first(["node2_log*.csv"], DATA_DIR),
    "node2_log_clean": find_first(["node2_log_clean*.csv"], DATA_DIR),

    "wind_h5": find_first(["*wind*.h5"], MODELS_DIR),
    "vestas_h5": find_first(["*vestas*.h5"], MODELS_DIR),

    "wind_tflite": find_first(["*wind*.tflite"], MODELS_DIR),
    "wind_int8_tflite": find_first(["*wind*int8*.tflite", "*wind*INT8*.tflite"], MODELS_DIR),
    "vestas_tflite": find_first(["*vestas*.tflite"], MODELS_DIR),
    "vestas_int8_tflite": find_first(["*vestas*int8*.tflite", "*vestas*INT8*.tflite"], MODELS_DIR),
    "node1_int8_tflite": find_first(["*node1*int8*.tflite", "*node1*.tflite"], MODELS_DIR),
    "node2_int8_tflite": find_first(["*node2*int8*.tflite", "*node2*.tflite"], MODELS_DIR),
}

In [20]:
path_check = pd.DataFrame({
    "key": list(PATHS.keys()),
    "full_path": [str(v) if v is not None else None for v in PATHS.values()],
    "exists": [v.exists() if v is not None else False for v in PATHS.values()]
})

path_check

,key,full_path,exists
0,wind_feature_description,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...,True
1,wind_event_info,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...,True
2,wind_dataset_dir,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...,True
3,vestas_excel,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\V...,True
4,pv_yield_excel,None,False
5,pv_fault_amb_mat,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\P...,True
6,pv_fault_elec_mat,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\P...,True
7,node1_log,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\l...,True
8,node1_log_clean,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\l...,True
9,node2_log,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\l...,True


In [21]:
path_check.to_csv(TABLES_DIR / "00_path_check.csv", index=False)
print("Saved:", TABLES_DIR / "00_path_check.csv")

Saved: C:\Users\MOINODHEEN\Moinu\Aegis_Project\tables\00_path_check.csv


In [22]:
wind_dataset_dir = PATHS["wind_dataset_dir"]

if wind_dataset_dir is None:
    raise ValueError("Wind dataset directory not found.")

wind_files = sorted(Path(wind_dataset_dir).glob("*.csv"), key=lambda x: x.name)

wind_file_table = pd.DataFrame({
    "file_name": [f.name for f in wind_files],
    "full_path": [str(f) for f in wind_files]
})

print("Wind CSV file count:", len(wind_files))
wind_file_table.head(10)

Wind CSV file count: 44


,file_name,full_path
0,0.csv,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...
1,10.csv,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...
2,13.csv,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...
3,14.csv,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...
4,17.csv,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...
5,22.csv,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...
6,24.csv,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...
7,25.csv,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...
8,26.csv,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...
9,3.csv,C:\Users\MOINODHEEN\Moinu\Aegis_Project\data\W...


In [23]:
wind_file_table.to_csv(TABLES_DIR / "00_wind_file_inventory.csv", index=False)
print("Saved:", TABLES_DIR / "00_wind_file_inventory.csv")

Saved: C:\Users\MOINODHEEN\Moinu\Aegis_Project\tables\00_wind_file_inventory.csv


In [24]:
def save_csv(df, filename, folder=TABLES_DIR, index=False):
    path = folder / filename
    df.to_csv(path, index=index)
    print("Saved CSV:", path)
    return path

def save_json(data, filename, folder=RESULTS_DIR):
    path = folder / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=str)
    print("Saved JSON:", path)
    return path

In [ ]:
RESEARCH_QUESTION = (
    "Can an edge-deployable autoencoder model detect wind SCADA anomalies effectively "
    "under different operating conditions, and can federated learning improve model "
    "robustness while keeping raw data local?"
)

run_metadata = {
    "project": "AEGIS",
    "created_from_notebook": "00_config",
    "timestamp": datetime.now().isoformat(),
    "seed": SEED,
    "project_root": str(PROJECT_ROOT),
    "research_question": RESEARCH_QUESTION,
    "pillars": [
        "Wind SCADA anomaly detection",
        "Cross-site and federated learning robustness",
        "ESP32 / TFLite edge deployment"
    ],
    "supporting_components_kept_in_project_record": [
        "Solar monitoring",
        "Renewable energy dashboard visualisation",
        "ESP32 edge node validation"
]
}

save_json(run_metadata, "00_run_metadata.json")
run_metadata

Saved JSON: C:\Users\MOINODHEEN\Moinu\Aegis_Project\results\00_run_metadata.json


{'project': 'AEGIS',
 'created_from_notebook': '00_config.ipynb',
 'timestamp': '2026-04-02T20:44:58.693240',
 'seed': 42,
 'project_root': 'C:\\Users\\MOINODHEEN\\Moinu\\Aegis_Project',
 'research_question': 'Can an edge-deployable autoencoder for wind SCADA anomaly detection retain useful performance under cross-site heterogeneity, and can federated training improve that robustness without raw-data sharing?',
 'pillars': ['Wind SCADA anomaly detection',
  'Cross-site and federated learning robustness',
  'ESP32 / TFLite edge deployment'],
 'supporting_components_kept_in_project_record': ['PV forecasting',
  'PV fault classification',
  'Dashboard visualisation']}

In [26]:
print("00_config complete.")

00_config complete.
